# Control clásico frente a aprendizaje por refuerzo en el péndulo invertido (CartPole-v1)

**Versión corta.** Mismo orden que la charla (**entorno → control clásico → aprendizaje por refuerzo →
comparación**), pero **sin entrenar ni sintonizar nada**: carga los parámetros, las políticas y las
métricas guardadas en `results/`, y solo simula episodios sueltos con controladores ya construidos. Se
ejecuta entero en menos de un minuto.

La versión completa, `01_charla_completa.ipynb`, recalcula en vivo la sintonización y una semilla de
cada entrenamiento. Las justificaciones de cada decisión están en `docs/`. Los números citados en el
texto proceden del registro y se indica su documento.

In [ ]:
# Importaciones y estilo comunes. Todo lo que no es trivial vive en el paquete cartpole_lab
# (probado en tests/), para que las celdas se lean durante la charla.
import json
import time

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, Markdown, display

from cartpole_lab import CartPoleTask, ForcePulse, continuous_dynamics, load_params, run_episode
from cartpole_lab import teaching as tl
from cartpole_lab.linearization import continuous_linearization, controllability_matrix, euler_discretization
from cartpole_lab.plotting import METHOD_COLORS, MUTED, SERIES, apply_style
from cartpole_lab.sanity import NEAR_BOUNDARY_INITIAL_STATES, REFERENCE_HARD_INITIAL_STATE, sanity_statistics

params = load_params()
apply_style()

## 1. El entorno: CartPole-v1 tal como lo simula Gymnasium (doc 01)

Estado $s = [x, \dot x, \theta, \dot\theta]$. El episodio termina si $|\theta| > 12°$ o $|x| > 2.4$
m, y se trunca a los 500 pasos (10 s). **Misma física, mismo estado, misma terminación y mismo bucle
para todos los métodos.** La única diferencia es el actuador: fuerza continua (clásicos y actor-crítico)
o 5 niveles (RL tabular y DQN).

In [ ]:
# Constantes físicas LEÍDAS del simulador (nunca copiadas a mano; docs/01 §2).
for name, value in vars(params).items():
    print(f"{name:<24} {value}")
print(f"\nθ límite = {np.degrees(params.theta_threshold_radians):.1f}°   ·   "
      f"τ = {params.tau} s ({1 / params.tau:.0f} Hz)   ·   episodio = 500 pasos = 10 s")

In [ ]:
# Convenciones de signo que dan por supuestas LQR y MPC (docs/01 §3), comprobadas sobre el modelo.
empuje = continuous_dynamics(np.zeros(4), +params.force_mag, params)
inclinado = continuous_dynamics([0.0, 0.0, 0.05, 0.0], 0.0, params)
print("F > 0 acelera el carro hacia +x (ẍ > 0):            ", bool(empuje[1] > 0))
print("F > 0 hace girar el poste hacia θ < 0 (θ̈ < 0):     ", bool(empuje[3] < 0))
print("Con F = 0, un θ > 0 crece (equilibrio inestable):   ", bool(inclinado[3] > 0))
display(Image(filename=str(tl.figure_path("00_convencion_signos.png")), width=720))

In [ ]:
# Linealización en el equilibrio vertical (docs/04 §2): de aquí salen el modo inestable y el modelo de LQR/MPC.
A_c, B_c = continuous_linearization(params)
A_d, B_d = euler_discretization(A_c, B_c, params.tau)
polo = float(np.max(np.linalg.eigvals(A_c).real))
print("autovalores de A_c:", np.round(np.sort(np.linalg.eigvals(A_c).real), 3))
print(f"modo inestable +√a = {polo:.2f} 1/s  →  sin control, un error se multiplica por e cada {1 / polo:.2f} s")
print("rango de la matriz de controlabilidad (Euler):", np.linalg.matrix_rank(controllability_matrix(A_d, B_d)), "de 4")

In [ ]:
# Sin control (F = 0) desde θ0 = 0.01 rad: ¿cuánto tarda en caer, y cuánto acierta el modelo lineal?
class FuerzaNula:
    """Controlador trivial (F = 0): fija la escala de tiempo del problema."""
    action_mode = "continuous"
    def reset(self): pass
    def __call__(self, state): return 0.0

s0 = np.array([0.0, 0.0, 0.01, 0.0])
caida = tl.simulate({"Sin control": FuerzaNula()}, initial_state=s0)["Sin control"]
lineal = np.array([np.linalg.matrix_power(A_d, k) @ s0 for k in range(caida.n_steps + 1)])
print(f"{caida.termination_reason} tras {caida.n_steps} pasos = {caida.duration:.2f} s")

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(caida.times, np.degrees(caida.states[:, 2]), color=SERIES[0], label="simulador (no lineal)")
ax.plot(caida.times, np.degrees(lineal[:, 2]), "--", color=SERIES[1], label="modelo lineal de Euler (LQR/MPC)")
ax.axhline(np.degrees(params.theta_threshold_radians), color=MUTED, linestyle=":")
ax.set_xlabel("t [s]"); ax.set_ylabel("θ [°]"); ax.legend(loc="upper left")
ax.set_title("Sin control el poste cae en menos de un segundo", loc="left", fontsize=11)
plt.show()

## 2. Control clásico

Parámetros cargados de `results/tuning/` (generados por `scripts/tune_pid.py`, `design_lqr.py`,
`design_mpc.py` y `tune_fuzzy.py`).

In [ ]:
pid_r, lqr_r = tl.load_result("tuning/pid.json"), tl.load_result("tuning/lqr.json")
mpc_r, fuz_r = tl.load_result("tuning/mpc.json"), tl.load_result("tuning/fuzzy.json")

print("PID en cascada (doc 03): evolución diferencial sobre ITAE + esfuerzo")
print("  ganancias:", {k: float(f"{v:.4g}") for k, v in pid_r["gains"].items()}, " → Ki ≈ 0: el mejor PID es un PD")
est = pid_r["diagnostics"]["stability"]
print(f"  constante de tiempo dominante (modos físicos): {est['dominant_time_constant_s']:.2f} s")

print("\nLQR (doc 04): modelo de Euler + pesos de Bryson")
print("  K =", np.round(lqr_r["K"][0], 3), f"  constante de tiempo dominante {lqr_r['dominant_time_constant_s']:.2f} s")
cruz = lqr_r["criteria_crossover_out_of_sample"]["paired"]
print(f"  cada uno con el criterio del otro (32 CI): el PID gana en ITAE en {cruz['pid_better_on_itae_count']}/32, "
      f"el LQR gana en su criterio en {cruz['lqr_better_on_lqr_criterion_count']}/32")

print("\nMPC (doc 05): restricciones explícitas, N = 25, Clarabel")
for nombre, v in mpc_r["near_boundary_comparison"].items():
    print(f"  {nombre}: sobrevive {sum(v)}/{len(v)} estados cerca de la frontera")
for solver, b in mpc_r["solver_benchmark"].items():
    print(f"  {solver:<24} peor caso {b['timing']['max_ms']:6.1f} ms frente a 20 ms de periodo; estados: {set(b['statuses'])}")

print("\nFuzzy Takagi-Sugeno (doc 06): 25 reglas fijas, 5 parámetros sintonizados como el PID")
pd = fuz_r["equivalent_pd"]
print(f"  PD equivalente en la celda central: Kp_θ = {pd['kp_theta']:.1f} N/rad, Kd_θ = {pd['kd_theta']:.1f} N·s/rad")
print(f"  ITAE fuera de muestra: {fuz_r['comparisons']['itae_out_of_sample']['fuzzy_vs_pid_relative_mean_pct']:.1f} % "
      f"peor que el PID de media (32 CI)")

In [ ]:
# Simulación en vivo (sin sintonizar): los 4 clásicos desde el estado difícil de referencia y desde
# un estado cerca de la frontera donde el LQR satura y pierde el poste.
clasicos = tl.method_controllers(params, ("PID", "LQR", "MPC", "Fuzzy"))
for s0, titulo in ((REFERENCE_HARD_INITIAL_STATE, "estado difícil de referencia"),
                   ((0.0, 0.0, 0.10, 1.25), "cerca de la frontera: poste inclinado y cayendo deprisa")):
    tl.plot_trajectories(tl.simulate(clasicos, initial_state=s0), params, title=f"Control clásico: {titulo}, s0 = {s0}")
tl.plot_force_maps({k: clasicos[k] for k in ("PID", "Fuzzy", "LQR")}, params,
                   title="Qué fuerza pide cada controlador con el carro centrado y quieto")
plt.show()

In [ ]:
for nombre in ("03_pid_respuesta.png", "04_lqr_vs_pid.png", "05_mpc_vs_lqr_pid.png", "06_fuzzy_reglas.png"):
    display(Image(filename=str(tl.figure_path(nombre)), width=850))

**Mensajes** (docs 03–06): Ziegler–Nichols no es aplicable al péndulo invertido; el PID se sintoniza
optimizando un criterio escrito. Cada método gana con su propio criterio. El MPC planifica con los
límites y recupera estados que el LQR pierde. El fuzzy es un lenguaje para escribir un PD con saturación
suave.

## 3. Aprendizaje por refuerzo

Políticas cargadas de `results/policies/` (la semilla exportada de cada método) y registros de
entrenamiento de `results/rl/`. Validación en las CI 2000–2009; test en las CI 1000–1049.

In [ ]:
tab, dqn_r, ac_r = tl.load_result("rl/tabular.json"), tl.load_result("rl/dqn.json"), tl.load_result("rl/actor_critic.json")

sel = tab["selection"]
print("Tabular (doc 07_08): discretización elegida por la regla declarada de antemano →", sel["chosen"])
for nombre, d in sel["per_discretization"].items():
    print(f"  {nombre:<11} {d['n_states']:>4} cajas: retorno greedy final {d['final_greedy_mean']:.0f} ± {d['final_greedy_std']:.0f}")
for metodo, nombre in (("q_learning", "Q-learning"), ("sarsa", "SARSA")):
    m = tab["methods"][metodo]
    print(f"  {nombre:<11} resuelto (media móvil ≥ 475) en {m['seeds_solved_classic']}/10 semillas; "
          f"test de la mejor política: sobrevive {m['test_best_checkpoint']['survival_rate']['mean']:.0%}, "
          f"esfuerzo {m['test_best_checkpoint']['effort_Ns']['mean']:.0f} N·s")

print("\nDQN y Double DQN (doc 09): 10 semillas cada uno")
for variante, nombre in (("dqn", "DQN"), ("double_dqn", "Double DQN")):
    r = dqn_r["summary"][variante]
    print(f"  {nombre:<11} validación final {r['final_greedy_validation']['mean']:.0f} frente a la mejor vista "
          f"{r['best_checkpoint_validation']['mean']:.0f} (colapso); Q(s = 0) final {r['q_at_rest_final']['mean']:.1f} (cota física 100)")

r = ac_r["summary"]
print("\nActor-crítico (doc 10): 10 semillas")
print(f"  resuelto en {r['seeds_solved_classic']}/10; test de la red final: sobrevive {r['test_final_network']['survival_rate']['mean']:.0%}, "
      f"esfuerzo {r['test_final_network']['effort_Ns']['mean']:.2f} N·s, |x| final {r['test_final_network']['last_second_max_abs_x_m']['mean']:.2f} m")
b = ac_r["boundary_checks_exported_seed"]
print(f"  fuera de distribución (12 estados cerca de la frontera): sobrevive {b['best']['near_boundary_survived']}/12 "
      f"(mejor red) y {b['final']['near_boundary_survived']}/12 (red final)")

In [ ]:
# Las políticas guardadas, en vivo: qué reglas aprendieron (mismo corte que los clásicos) y cómo responden.
rl = tl.method_controllers(params, ("Q-learning", "SARSA", "DQN", "Actor-crítico"))
tl.plot_force_maps({**{k: rl[k] for k in ("Q-learning", "DQN", "Actor-crítico")},
                    "Fuzzy (escrito)": clasicos["Fuzzy"]}, params,
                   title="Reglas aprendidas frente a reglas escritas (carro centrado y quieto)")
tl.plot_trajectories(tl.simulate({"LQR": clasicos["LQR"], **rl}, seed=1000), params,
                     title="RL frente a LQR en una CI de la distribución oficial (semilla 1000)")
plt.show()

In [ ]:
for nombre in ("08_curvas_tabulares.png", "09_curvas_dqn.png", "09_diagnostico_dqn.png", "10_actor_critico_vs_clasicos.png"):
    display(Image(filename=str(tl.figure_path(nombre)), width=850))

**Mensajes** (docs 07_08, 09 y 10): el RL tabular aprende sin modelo pero no converge y controla en
*bang-bang*. El DQN aprende rápido pero colapsa; la sobreestimación no era la causa. El actor-crítico
converge y redescubre un PD saturado, pero solo sabe lo que ha vivido. Todos hacen lo que pide la
recompensa (sobrevivir), no lo que queremos (centrar el carro).

## 4. Comparación (docs/12 y 13)

Protocolo pre-registrado; RL con media ± desviación típica entre 10 semillas.

In [ ]:
# Resultados de la evaluación pre-registrada (docs/12): 8 métodos, 50 CI nominales, 8 impulsos,
# 2 mallas de 21 × 21 estados iniciales; RL con 10 semillas por método.
comparacion = tl.load_result("metrics/comparison.json")
print(f"{comparacion['n_jobs']} tareas · protocolo: {comparacion['protocol']} · "
      f"impulsos {comparacion['impulses_Ns']} N·s · malla {comparacion['grid_n']} × {comparacion['grid_n']}")
display(Markdown(tl.summary_table_markdown()))

In [ ]:
# Figuras finales (scripts/make_figures.py), generadas a partir de comparison.json.
for nombre in ("13_resumen_nominal.png", "13_robustez_impulso.png", "13_regiones_atraccion.png",
               "13_coste_aprendizaje.png"):
    display(Image(filename=str(tl.figure_path(nombre)), width=1000))

In [ ]:
# Demostración en vivo del escenario de robustez (docs/12 §2): misma CI (semilla 1000, distribución
# oficial) y un impulso lateral de 1.0 N·s en t = 5 s para los 8 métodos. RL: la semilla exportada.
from cartpole_lab.evaluation.metrics import episode_metrics
from cartpole_lab.evaluation.protocol import IMPULSE_STEP

J = 1.0  # N·s: por debajo del J₅₀ de los cuatro clásicos en el registro (1.25–1.5 N·s, doc 13)
tarea_impulso = CartPoleTask(action_mode="continuous", disturbance=ForcePulse(IMPULSE_STEP, 1, J / params.tau))
controladores = tl.method_controllers(params)
t0 = time.perf_counter()
try:
    tray_impulso = tl.simulate(controladores, seed=1000, task=tarea_impulso)
finally:
    tarea_impulso.close()
print(f"8 episodios en {time.perf_counter() - t0:.1f} s\n")
print(f"{'método':<15}{'sobrevive':>10}{'recuperación del poste [s]':>30}{'esfuerzo [N·s]':>17}")
for nombre, tr in tray_impulso.items():
    m = episode_metrics(tr, impulse_step=IMPULSE_STEP)
    rec = "—" if m["recovery_time_s"] is None else f"{m['recovery_time_s']:.2f}"
    print(f"{nombre:<15}{str(m['survived']):>10}{rec:>30}{m['effort_abs_Ns']:>17.2f}")

t_imp = IMPULSE_STEP * params.tau
clasicos = ("PID", "LQR", "MPC", "Fuzzy")
tl.plot_trajectories({k: v for k, v in tray_impulso.items() if k in clasicos}, params, impulse_time=t_imp,
                     title=f"Control clásico: impulso de {J} N·s en t = {t_imp:.0f} s")
tl.plot_trajectories({k: v for k, v in tray_impulso.items() if k not in clasicos}, params, impulse_time=t_imp,
                     title="Aprendizaje por refuerzo (semilla exportada): mismo impulso")
plt.show()

### Mensajes para la charla (docs/13 §8, sobre la evaluación registrada)

1. **Si hay modelo, úsalo.** LQR y MPC controlan con precisión, gastando muy poco y sin un solo
   episodio de prueba. El MPC además amplía la región de recuperación, porque planifica con las
   restricciones.
2. **El RL aprende a no caerse, no a controlar.** Q-learning, SARSA y DQN mantienen el poste arriba
   oscilando, con 200 veces más esfuerzo. El actor-crítico se acerca a un clásico en el poste, pero deja
   el carro donde caiga, porque la recompensa no lo pide.
3. **El RL solo sabe lo que ha vivido.** Fuera de la distribución de entrenamiento, sus regiones de
   recuperación son de 1.5 a 10 veces menores que las de los clásicos.
4. **Cuidado con cómo se mide la robustez.** Que una política sobreviva a un impulso mayor puede ser
   suerte de fase, no robustez (doc 13 §3).
5. **Cuidado con cómo se mide la eficiencia.** Un PID "clásico" sintonizado por fuerza bruta usó más
   simulaciones que cualquier RL (doc 13 §5).

Limitaciones de la comparación (horizonte de 10 s, un solo tipo de perturbación, cortes 2D de la región
de atracción, 10 semillas…): docs/13 §9.